<a href="https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Prioritize the pages that have **meaningful search visibility** but are receiving **fewer clicks than expected for their average search position**.

The baseline will rank pages using two observable signals:

* **CTR relative to position** — identifies pages that appear to under-capture clicks compared with other pages at similar positions.
* **Impressions** — ensures that the opportunity is based on meaningful search visibility rather than a small amount of noisy data.

The output is a **review-priority ranking**, not a prediction that changing the page will improve its performance.

### Reason codes

* `low_ctr_visible_page` — the page has meaningful search visibility but its CTR is weak relative to pages at a similar search position.
* `insufficient_visibility` — the page does not have enough impressions for a reliable CTR-based review.
* `position_unavailable` — average position is unavailable or zero, so a position-adjusted CTR comparison cannot be made.
* `not_prioritized` — the page does not meet the conditions for the baseline review queue.

### Signal check 1 — CTR relative to position

**Signal:** CTR compared within average-position groups.

CTR should not be judged against one global threshold because pages ranking near position 1 naturally receive different click rates from pages ranking near position 10. We thus compare CTR within position tiers and check whether lower relative CTR is associated with the observed outcome.=

**Verdict:** based on the observed bucket table produced in the audit.

### Signal check 2 — Search visibility

**Signal:** GSC impressions.

Impressions provide the visibility context for the CTR signal. A low CTR based on very few impressions may simply be noise, so the rule should give greater priority to pages with meaningful existing visibility.


In [1]:
# --- Setup: connect DuckDB to the Hugging Face warehouse ---

import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Read the HF token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs; LOAD httpfs;")

# Register the token as a DuckDB secret
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
)
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

FACT_TABLE_GLOB = (
    f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"
)

print("DuckDB connected and Hugging Face secret registered.")

DuckDB connected and Hugging Face secret registered.


In [2]:
march_path = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print(march_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [3]:
march_data = con.sql(f"""SELECT * FROM read_parquet('{march_path}')""").df()

print("Rows:", len(march_data))
print("Columns:", len(march_data.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Columns: 31


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
""").df()

df["report_date"] = pd.to_datetime(df["report_date"])

print(f"Total rows loaded: {len(df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows loaded: 3,611,061


2 time windows

In [5]:
feature_df = df[
    df["report_date"].dt.day <= 15
].copy()

outcome_df = df[
    df["report_date"].dt.day >= 16
].copy()

print("Information-window rows:", len(feature_df))
print("Outcome-window rows:", len(outcome_df))

Information-window rows: 1640237
Outcome-window rows: 1970824


page level feature table

In [16]:
features = (
    feature_df
    .groupby(["client_hash_id", "content_hash_id"])
    .agg(
        impressions=("gsc_impressions", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        sessions_organic=("sessions_organic", "sum"),
        ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
        clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

features["ctr"] = np.where(
    features["impressions"] > 0,
    features["clicks"] / features["impressions"],
    np.nan
)

signal check 1

In [17]:
signal_df = features[
    (features["impressions"] > 0) &
    (features["ctr"].notna()) &
    (features["avg_position"].notna()) &
    (features["avg_position"] > 0)
].copy()

def position_bucket(position):
    if position <= 3:
        return "1-3"
    elif position <= 5:
        return "4-5"
    elif position <= 10:
        return "6-10"
    elif position <= 20:
        return "11-20"
    else:
        return "21+"

signal_df["position_bucket"] = signal_df["avg_position"].apply(
    position_bucket
)

ctr_signal_check = (
    signal_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

display(ctr_signal_check)

,position_bucket,n,median_ctr,mean_ctr
0,1-3,16251,0.000000,0.009315
1,11-20,26912,0.000000,0.003337
2,21+,37511,0.000000,0.002096
3,4-5,23375,0.000546,0.006055
4,6-10,46626,0.000000,0.003952


In [18]:
expected_ctr = (
    signal_df
    .groupby("position_bucket", observed=False)["ctr"]
    .median()
    .rename("expected_ctr")
)

signal_df = signal_df.merge(
    expected_ctr,
    on="position_bucket",
    how="left"
)

signal_df["ctr_gap"] = (
    signal_df["expected_ctr"] - signal_df["ctr"]
)

signal_df["relative_ctr"] = (
    signal_df["ctr"] / signal_df["expected_ctr"]
)

signal check 2

In [22]:
visibility_check = (
    features[features["impressions"] > 50]
    .assign(
        visibility_bucket=lambda x: pd.cut(
            x["impressions"],
            bins=[0, 100, 500, 1000, 5000, np.inf],
            labels=[
                "1-100",
                "101-500",
                "501-1,000",
                "1,001-5,000",
                "5,000+"
            ]
        )
    )
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

display(visibility_check)

,visibility_bucket,n,median_impressions
0,1-100,14841,72.0
1,101-500,35526,223.0
2,"501-1,000",14799,713.0
3,"1,001-5,000",21544,1887.0
4,"5,000+",5423,7975.0


building obbserved outcome

In [ ]:
first_half_impressions = (
    feature_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        first_half_impressions=("gsc_impressions", "sum")
    )
)

second_half_impressions = (
    outcome_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        second_half_impressions=("gsc_impressions", "sum")
    )
)

outcomes = first_half_impressions.merge(
    second_half_impressions,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

outcomes["is_declining"] = (
    outcomes["second_half_impressions"]
    < outcomes["first_half_impressions"]
)

display(outcomes.head())

ctr signal

In [ ]:
ctr_audit = signal_df.merge(
    outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

ctr_audit["low_relative_ctr"] = (
    ctr_audit["relative_ctr"] < 1
)

ctr_verdict_table = (
    ctr_audit
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("is_declining", "mean"),
        median_relative_ctr=("relative_ctr", "median")
    )
    .reset_index()
)

display(ctr_verdict_table)

visibility signal

In [ ]:
visibility_audit = features.merge(
    outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

visibility_audit["visibility_bucket"] = pd.cut(
    visibility_audit["impressions"],
    bins=[0, 100, 500, 1000, 5000, np.inf],
    labels=[
        "1-100",
        "101-500",
        "501-1,000",
        "1,001-5,000",
        "5,000+"
    ]
)

visibility_verdict_table = (
    visibility_audit
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("is_declining", "mean")
    )
    .reset_index()
)

display(visibility_verdict_table)

baseline score


In [ ]:
scoring = signal_df.copy()

# Pages with enough observable visibility are eligible.
MIN_IMPRESSIONS = 100

scoring["eligible"] = (
    (scoring["impressions"] >= MIN_IMPRESSIONS) &
    (scoring["expected_ctr"] > 0)
)

scoring = scoring[scoring["eligible"]].copy()

# Relative CTR shortfall.
scoring["ctr_shortfall"] = (
    1 - scoring["relative_ctr"]
).clip(lower=0)

# Visibility-weighted opportunity score.
scoring["score"] = (
    np.log1p(scoring["impressions"])
    * scoring["ctr_shortfall"]
)

reason code and action

In [ ]:
scoring["reason_code"] = "low_ctr_visible_page"
scoring["action"] = "review_ctr"

ranking

In [ ]:
queue = (
    scoring
    .sort_values(
        ["score", "impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)

display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "expected_ctr",
            "ctr_shortfall"
        ]
    ].head(10)
)

precision at k

In [ ]:
evaluation = queue.merge(
    outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

base_rate = evaluation["is_declining"].mean()

print(f"Observed decline base rate: {base_rate:.3f}")

In [ ]:
for k in [10, 50, 100]:
    top_k = evaluation.head(k)

    precision_at_k = top_k["is_declining"].mean()

    print(
        f"Precision@{k}: "
        f"{precision_at_k:.3f} "
        f"({top_k['is_declining'].sum()}/{len(top_k)})"
    )

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.